# 🚀 Gender + Age Model - Ultra Optimized Training

**Mục tiêu:**
- Model size: 84MB → ~5-10MB
- Speed: 200ms → ~20-30ms per face
- Gender accuracy: 95% → ≥97%
- Age MAE: 5-6 years → ≤4 years

**Kỹ thuật sử dụng:**
1. MobileNetV3-Large backbone
2. Knowledge Distillation từ EfficientNetV2 teacher
3. Label Distribution Learning cho age
4. Multi-task learning với attention
5. Advanced data augmentation
6. Mixed Precision Training (AMP)
7. Export ONNX + INT8 quantization

## 1. Setup & Imports

In [ ]:
# Install required packages
!pip install -q timm albumentations onnx onnxruntime

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.cuda.amp import GradScaler, autocast
import timm
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
import copy
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
class Config:
    # Paths - ADJUST THESE FOR YOUR KAGGLE DATASET
    DATA_DIR = "/kaggle/input/utkface-new/UTKFace"  # UTKFace dataset path
    TEACHER_PATH = "/kaggle/input/gender-age-model/gender_age_model.pth"  # Your current model
    OUTPUT_DIR = "/kaggle/working"
    
    # Model
    STUDENT_BACKBONE = "mobilenetv3_large_100"  # Lightweight, fast
    TEACHER_BACKBONE = "tf_efficientnetv2_s"    # Current model backbone
    IMG_SIZE = 160  # Smaller for speed (was 224)
    NUM_AGES = 10   # Age groups
    
    # Training
    EPOCHS = 50
    BATCH_SIZE = 64
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 4
    
    # Mixed Precision
    USE_AMP = True  # Faster training on GPU
    
    # Early Stopping
    PATIENCE = 10  # Stop if no improvement for 10 epochs
    
    # Knowledge Distillation
    TEMPERATURE = 4.0
    ALPHA = 0.7  # Weight for distillation loss
    
    # Label Distribution Learning
    LDL_SIGMA = 1.5  # Gaussian spread for age distribution
    
    # Age groups mapping
    AGE_GROUPS = ['0-2', '3-6', '7-12', '13-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
    AGE_BOUNDARIES = [0, 3, 7, 13, 18, 25, 35, 45, 55, 65, 120]

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

## 3. Label Distribution Learning (LDL)

In [ ]:
def age_to_group(age: int) -> int:
    """Convert age to group index (0-9)"""
    for i, (low, high) in enumerate(zip(cfg.AGE_BOUNDARIES[:-1], cfg.AGE_BOUNDARIES[1:])):
        if low <= age < high:
            return i
    return cfg.NUM_AGES - 1


def create_age_distribution(age: int, num_classes: int = 10, sigma: float = 1.5) -> np.ndarray:
    """
    Create soft label distribution for age using Gaussian.
    This is Label Distribution Learning (LDL) - allows model to learn
    that adjacent ages are related.
    
    Research: LDL reduces MAE from ~5 to ~2 on benchmark datasets.
    """
    target_group = age_to_group(age)
    
    # Create Gaussian distribution centered at target group
    groups = np.arange(num_classes)
    distribution = np.exp(-0.5 * ((groups - target_group) / sigma) ** 2)
    
    # Normalize to sum to 1
    distribution = distribution / distribution.sum()
    
    return distribution.astype(np.float32)


# Test LDL
print("Age 25 (group 5) distribution:")
print(create_age_distribution(25))
print("\nAge 5 (group 1) distribution:")
print(create_age_distribution(5))

## 4. Dataset with Advanced Augmentation

In [ ]:
def get_train_transforms(img_size: int):
    """
    Advanced augmentation pipeline.
    Research: Reduces MAE by 0.26-0.41, improves generalization.
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        
        # Geometric transforms
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
        A.Affine(shear=(-10, 10), p=0.3),
        
        # Color transforms
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=1),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1),
        ], p=0.5),
        
        # Noise & blur
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50), p=1),
            A.GaussianBlur(blur_limit=(3, 5), p=1),
            A.MotionBlur(blur_limit=5, p=1),
        ], p=0.2),
        
        # Occlusion (simulates partial face visibility)
        A.CoarseDropout(max_holes=1, max_height=int(img_size*0.2), max_width=int(img_size*0.2),
                        min_holes=1, min_height=int(img_size*0.1), min_width=int(img_size*0.1),
                        fill_value=0, p=0.2),
        
        # Normalize
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_val_transforms(img_size: int):
    """Simple validation transforms"""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

In [ ]:
class UTKFaceDataset(Dataset):
    """
    UTKFace dataset with Label Distribution Learning.
    Filename format: age_gender_race_date.jpg
    """
    
    def __init__(self, root_dir: str, transform=None, use_ldl: bool = True, indices=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.use_ldl = use_ldl
        
        self.samples = []
        
        # Parse filenames
        all_files = list(self.root_dir.glob("*.jpg"))
        for img_path in all_files:
            try:
                parts = img_path.stem.split("_")
                age = int(parts[0])
                gender = int(parts[1])  # 0=male, 1=female
                
                if 0 <= age <= 116 and gender in [0, 1]:
                    self.samples.append({
                        'path': str(img_path),
                        'age': age,
                        'gender': gender,
                        'age_group': age_to_group(age)
                    })
            except (ValueError, IndexError):
                continue
        
        # Filter by indices if provided (for train/val split)
        if indices is not None:
            self.samples = [self.samples[i] for i in indices]
        
        print(f"Loaded {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load image
        image = np.array(Image.open(sample['path']).convert('RGB'))
        
        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        # Labels
        gender = torch.tensor(sample['gender'], dtype=torch.long)
        age_group = torch.tensor(sample['age_group'], dtype=torch.long)
        
        # Label Distribution for age (soft labels)
        if self.use_ldl:
            age_dist = torch.tensor(
                create_age_distribution(sample['age'], cfg.NUM_AGES, cfg.LDL_SIGMA),
                dtype=torch.float32
            )
        else:
            age_dist = age_group
        
        return image, gender, age_group, age_dist

## 5. Model Architecture

In [ ]:
class ChannelAttention(nn.Module):
    """Channel attention module for multi-task learning"""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class GenderAgeStudent(nn.Module):
    """
    Lightweight student model with MobileNetV3 backbone.
    Uses channel attention for multi-task learning.
    
    Research basis:
    - MobileNetV3: 2-3x faster than EfficientNetV2
    - Multi-task attention: improves gender acc to 99%
    """
    
    def __init__(self, backbone: str = 'mobilenetv3_large_100', num_ages: int = 10):
        super().__init__()
        
        # Backbone
        self.backbone = timm.create_model(backbone, pretrained=True, features_only=True)
        self.num_features = self.backbone.feature_info[-1]['num_chs']
        
        # Attention for shared features
        self.attention = ChannelAttention(self.num_features)
        
        # Global pooling
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Shared feature extraction
        self.shared_fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
        )
        
        # Task-specific heads
        self.gender_head = nn.Linear(256, 2)
        self.age_head = nn.Linear(256, num_ages)
    
    def forward(self, x):
        # Extract features (last feature map)
        features = self.backbone(x)[-1]
        
        # Apply attention
        features = self.attention(features)
        
        # Pool and flatten
        features = self.pool(features).flatten(1)
        
        # Shared features
        shared = self.shared_fc(features)
        
        # Task outputs
        gender = self.gender_head(shared)
        age = self.age_head(shared)
        
        return gender, age


# Test model
model = GenderAgeStudent(cfg.STUDENT_BACKBONE, cfg.NUM_AGES)
x = torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE)
g, a = model(x)
print(f"Gender output: {g.shape}")
print(f"Age output: {a.shape}")
print(f"Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
class GenderAgeTeacher(nn.Module):
    """
    Teacher model (your current EfficientNetV2 model).
    Used to guide the student model via Knowledge Distillation.
    """
    
    def __init__(self, backbone: str = 'tf_efficientnetv2_s', num_ages: int = 10):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)
        n = self.backbone.num_features
        
        self.fc = nn.Sequential(
            nn.BatchNorm1d(n),
            nn.Dropout(0.3),
            nn.Linear(n, 512),
            nn.ReLU()
        )
        self.gender = nn.Linear(512, 2)
        self.age = nn.Linear(512, num_ages)
    
    def forward(self, x):
        f = self.fc(self.backbone(x))
        return self.gender(f), self.age(f)


def load_teacher_model(checkpoint_path: str, device):
    """Load pre-trained teacher model"""
    teacher = GenderAgeTeacher(cfg.TEACHER_BACKBONE, cfg.NUM_AGES)
    
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        teacher.load_state_dict(checkpoint['model_state_dict'])
        print("✅ Teacher model loaded from checkpoint")
    else:
        print("⚠️ Teacher checkpoint not found, training without distillation")
        return None
    
    teacher.eval()
    for param in teacher.parameters():
        param.requires_grad = False
    
    return teacher.to(device)

## 6. Knowledge Distillation Loss

In [ ]:
class DistillationLoss(nn.Module):
    """
    Combined loss for Knowledge Distillation + Label Distribution Learning.
    
    Research basis:
    - KD improves gender acc from 88% to 90.78%
    - LDL reduces age MAE from ~5 to ~2 years
    """
    
    def __init__(self, temperature: float = 4.0, alpha: float = 0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha  # Weight for distillation loss
        
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_loss = nn.KLDivLoss(reduction='batchmean')
    
    def forward(self, 
                student_gender, student_age,
                teacher_gender, teacher_age,
                target_gender, target_age_group, target_age_dist):
        
        # === Gender Loss ===
        gender_hard_loss = self.ce_loss(student_gender, target_gender)
        
        if teacher_gender is not None:
            gender_soft_student = F.log_softmax(student_gender / self.temperature, dim=1)
            gender_soft_teacher = F.softmax(teacher_gender / self.temperature, dim=1)
            gender_distill_loss = self.kl_loss(gender_soft_student, gender_soft_teacher) * (self.temperature ** 2)
            gender_loss = (1 - self.alpha) * gender_hard_loss + self.alpha * gender_distill_loss
        else:
            gender_loss = gender_hard_loss
        
        # === Age Loss (with LDL) ===
        age_soft_student = F.log_softmax(student_age, dim=1)
        age_ldl_loss = self.kl_loss(age_soft_student, target_age_dist)
        
        if teacher_age is not None:
            age_soft_teacher = F.softmax(teacher_age / self.temperature, dim=1)
            age_distill_loss = self.kl_loss(age_soft_student, age_soft_teacher) * (self.temperature ** 2)
            age_loss = (1 - self.alpha) * age_ldl_loss + self.alpha * age_distill_loss
        else:
            age_loss = age_ldl_loss
        
        total_loss = gender_loss + age_loss
        return total_loss, gender_loss.item(), age_loss.item()

## 7. Training Loop with AMP

In [ ]:
def train_one_epoch(model, teacher, train_loader, optimizer, criterion, scaler, device, use_amp):
    model.train()
    
    total_loss = 0
    gender_correct = 0
    age_correct = 0
    total_samples = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for images, genders, age_groups, age_dists in pbar:
        images = images.to(device)
        genders = genders.to(device)
        age_groups = age_groups.to(device)
        age_dists = age_dists.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        with autocast(enabled=use_amp):
            student_gender, student_age = model(images)
            
            if teacher is not None:
                with torch.no_grad():
                    teacher_gender, teacher_age = teacher(images)
            else:
                teacher_gender, teacher_age = None, None
            
            loss, g_loss, a_loss = criterion(
                student_gender, student_age,
                teacher_gender, teacher_age,
                genders, age_groups, age_dists
            )
        
        # Backward with scaler
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        # Metrics
        total_loss += loss.item() * images.size(0)
        gender_correct += (student_gender.argmax(1) == genders).sum().item()
        age_correct += (student_age.argmax(1) == age_groups).sum().item()
        total_samples += images.size(0)
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'g_acc': f'{gender_correct/total_samples:.3f}',
            'a_acc': f'{age_correct/total_samples:.3f}'
        })
    
    return {
        'loss': total_loss / total_samples,
        'gender_acc': gender_correct / total_samples,
        'age_acc': age_correct / total_samples
    }

In [ ]:
@torch.no_grad()
def validate(model, val_loader, device):
    model.eval()
    
    gender_correct = 0
    age_correct = 0
    age_mae = 0
    total_samples = 0
    
    for images, genders, age_groups, _ in tqdm(val_loader, desc="Validation"):
        images = images.to(device)
        genders = genders.to(device)
        age_groups = age_groups.to(device)
        
        pred_gender, pred_age = model(images)
        
        gender_correct += (pred_gender.argmax(1) == genders).sum().item()
        age_correct += (pred_age.argmax(1) == age_groups).sum().item()
        age_mae += torch.abs(pred_age.argmax(1).float() - age_groups.float()).sum().item()
        total_samples += images.size(0)
    
    return {
        'gender_acc': gender_correct / total_samples,
        'age_acc': age_correct / total_samples,
        'age_mae_groups': age_mae / total_samples
    }

## 8. Main Training with Early Stopping

In [ ]:
def create_datasets():
    """Create train and val datasets with proper separation"""
    # First, get all valid samples
    temp_dataset = UTKFaceDataset(cfg.DATA_DIR, transform=None, use_ldl=True)
    n_samples = len(temp_dataset.samples)
    
    # Create indices for split
    np.random.seed(42)
    indices = np.random.permutation(n_samples)
    train_size = int(0.9 * n_samples)
    train_indices = indices[:train_size].tolist()
    val_indices = indices[train_size:].tolist()
    
    # Create separate datasets with proper transforms
    train_dataset = UTKFaceDataset(
        cfg.DATA_DIR,
        transform=get_train_transforms(cfg.IMG_SIZE),
        use_ldl=True,
        indices=train_indices
    )
    
    val_dataset = UTKFaceDataset(
        cfg.DATA_DIR,
        transform=get_val_transforms(cfg.IMG_SIZE),
        use_ldl=True,
        indices=val_indices
    )
    
    return train_dataset, val_dataset

In [ ]:
def train():
    print("Loading datasets...")
    train_dataset, val_dataset = create_datasets()
    
    train_loader = DataLoader(
        train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=cfg.NUM_WORKERS, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=cfg.NUM_WORKERS, pin_memory=True
    )
    
    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
    
    # Models
    student = GenderAgeStudent(cfg.STUDENT_BACKBONE, cfg.NUM_AGES).to(device)
    teacher = load_teacher_model(cfg.TEACHER_PATH, device)
    
    # Training setup
    criterion = DistillationLoss(cfg.TEMPERATURE, cfg.ALPHA)
    optimizer = AdamW(student.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    # Early stopping
    best_gender_acc = 0
    patience_counter = 0
    history = []
    
    for epoch in range(cfg.EPOCHS):
        print(f"\nEpoch {epoch+1}/{cfg.EPOCHS}")
        
        train_metrics = train_one_epoch(
            student, teacher, train_loader, optimizer, criterion,
            scaler, device, cfg.USE_AMP
        )
        val_metrics = validate(student, val_loader, device)
        scheduler.step()
        
        # Log
        print(f"Train - Loss: {train_metrics['loss']:.4f}, Gender: {train_metrics['gender_acc']:.3f}, Age: {train_metrics['age_acc']:.3f}")
        print(f"Val   - Gender: {val_metrics['gender_acc']:.3f}, Age: {val_metrics['age_acc']:.3f}, MAE: {val_metrics['age_mae_groups']:.2f}")
        
        history.append({
            'epoch': epoch + 1,
            'train_loss': train_metrics['loss'],
            'train_gender_acc': train_metrics['gender_acc'],
            'val_gender_acc': val_metrics['gender_acc'],
            'val_age_acc': val_metrics['age_acc'],
        })
        
        # Save best & early stopping
        if val_metrics['gender_acc'] > best_gender_acc:
            best_gender_acc = val_metrics['gender_acc']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': student.state_dict(),
                'gender_acc': val_metrics['gender_acc'],
                'age_acc': val_metrics['age_acc'],
                'backbone': cfg.STUDENT_BACKBONE,
                'img_size': cfg.IMG_SIZE,
                'num_ages': cfg.NUM_AGES,
            }, f"{cfg.OUTPUT_DIR}/best_model.pth")
            print(f"💾 Saved best (gender_acc: {best_gender_acc:.3f})")
        else:
            patience_counter += 1
            if patience_counter >= cfg.PATIENCE:
                print(f"\n⚠️ Early stopping at epoch {epoch+1}")
                break
    
    # Load best model
    checkpoint = torch.load(f"{cfg.OUTPUT_DIR}/best_model.pth")
    student.load_state_dict(checkpoint['model_state_dict'])
    print(f"\n✅ Training complete! Best gender_acc: {checkpoint['gender_acc']:.3f}")
    
    return student, history

In [ ]:
# Run training
model, history = train()

## 9. Plot Training History

In [ ]:
def plot_history(history):
    df = pd.DataFrame(history)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss
    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].set_title('Training Loss')
    
    # Accuracy
    axes[1].plot(df['epoch'], df['train_gender_acc'], label='Train Gender')
    axes[1].plot(df['epoch'], df['val_gender_acc'], label='Val Gender')
    axes[1].plot(df['epoch'], df['val_age_acc'], label='Val Age', linestyle='--')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].set_title('Accuracy')
    
    plt.tight_layout()
    plt.savefig(f"{cfg.OUTPUT_DIR}/training_history.png", dpi=150)
    plt.show()

plot_history(history)

## 10. Export to ONNX

In [ ]:
def export_onnx(model, output_path: str, img_size: int = 160):
    model.eval()
    model.cpu()
    
    dummy_input = torch.randn(1, 3, img_size, img_size)
    
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        input_names=['input'],
        output_names=['gender', 'age'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'gender': {0: 'batch_size'},
            'age': {0: 'batch_size'}
        },
        opset_version=13
    )
    
    print(f"✅ Exported: {output_path}")
    print(f"   Size: {os.path.getsize(output_path) / 1e6:.2f} MB")

export_onnx(model, f"{cfg.OUTPUT_DIR}/gender_age_optimized.onnx", cfg.IMG_SIZE)

## 11. ONNX Quantization (INT8)

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

def quantize_onnx(input_path: str, output_path: str):
    quantize_dynamic(
        input_path,
        output_path,
        weight_type=QuantType.QUInt8
    )
    
    original_size = os.path.getsize(input_path) / 1e6
    quantized_size = os.path.getsize(output_path) / 1e6
    
    print(f"✅ Quantized: {output_path}")
    print(f"   Original: {original_size:.2f} MB → Quantized: {quantized_size:.2f} MB")
    print(f"   Reduction: {(1 - quantized_size/original_size)*100:.1f}%")

quantize_onnx(
    f"{cfg.OUTPUT_DIR}/gender_age_optimized.onnx",
    f"{cfg.OUTPUT_DIR}/gender_age_optimized_int8.onnx"
)

## 12. Benchmark

In [ ]:
import onnxruntime as ort
import time

def benchmark_onnx(onnx_path: str, img_size: int = 160, num_runs: int = 100):
    session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    dummy_input = np.random.randn(1, 3, img_size, img_size).astype(np.float32)
    
    # Warmup
    for _ in range(10):
        session.run(None, {'input': dummy_input})
    
    # Benchmark
    times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        session.run(None, {'input': dummy_input})
        times.append((time.perf_counter() - start) * 1000)
    
    avg = np.mean(times)
    print(f"  {Path(onnx_path).name}: {avg:.2f} ms (FPS: {1000/avg:.1f})")
    return avg

print("\n" + "="*50)
print("BENCHMARK (CPU)")
print("="*50)

fp32 = benchmark_onnx(f"{cfg.OUTPUT_DIR}/gender_age_optimized.onnx", cfg.IMG_SIZE)
int8 = benchmark_onnx(f"{cfg.OUTPUT_DIR}/gender_age_optimized_int8.onnx", cfg.IMG_SIZE)

print(f"\n🚀 INT8 is {fp32/int8:.2f}x faster")

## 13. Download Files

In [ ]:
print("\n" + "="*50)
print("📦 OUTPUT FILES")
print("="*50)

for f in sorted(os.listdir(cfg.OUTPUT_DIR)):
    path = os.path.join(cfg.OUTPUT_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e6
        print(f"  {f}: {size:.2f} MB")

print("\n✅ Download và copy vào project:")
print("  1. best_model.pth - PyTorch checkpoint")
print("  2. gender_age_optimized_int8.onnx - ONNX optimized (RECOMMENDED)")
print("\n🎯 Rename gender_age_optimized_int8.onnx → gender_age_optimized_int8.onnx")
print("   và copy vào root folder của project")

## 14. Confusion Matrix Visualization

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

def plot_confusion_matrices(model, val_loader, device):
    model.eval()
    all_gender_preds, all_gender_true = [], []
    all_age_preds, all_age_true = [], []
    with torch.no_grad():
        for images, genders, age_groups, _ in tqdm(val_loader):
            images = images.to(device)
            pred_gender, pred_age = model(images)
            all_gender_preds.extend(pred_gender.argmax(1).cpu().numpy())
            all_gender_true.extend(genders.numpy())
            all_age_preds.extend(pred_age.argmax(1).cpu().numpy())
            all_age_true.extend(age_groups.numpy())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cm_gender = confusion_matrix(all_gender_true, all_gender_preds)
    sns.heatmap(cm_gender, annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_title("Gender Confusion Matrix")
    cm_age = confusion_matrix(all_age_true, all_age_preds)
    sns.heatmap(cm_age, annot=True, fmt="d", cmap="Oranges", ax=axes[1])
    axes[1].set_title("Age Group Confusion Matrix")
    plt.savefig(f"{cfg.OUTPUT_DIR}/confusion_matrices.png", dpi=150)
    plt.show()
    print(classification_report(all_gender_true, all_gender_preds, target_names=["Male", "Female"]))
    print(classification_report(all_age_true, all_age_preds, target_names=cfg.AGE_GROUPS))
    return all_gender_preds, all_gender_true, all_age_preds, all_age_true

_, val_dataset = create_datasets()
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)
gender_preds, gender_true, age_preds, age_true = plot_confusion_matrices(model, val_loader, device)

## 15. Age Distribution Analysis

In [ ]:
def plot_age_distribution(age_true, age_preds):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(range(len(cfg.AGE_GROUPS)), np.bincount(age_true, minlength=10))
    axes[0].set_xticklabels(cfg.AGE_GROUPS, rotation=45)
    axes[0].set_title("Dataset Distribution")
    errors = np.array(age_preds) - np.array(age_true)
    axes[2].hist(errors, bins=range(-5, 6), color="salmon")
    axes[2].set_title("Error Distribution")
    plt.savefig(f"{cfg.OUTPUT_DIR}/age_distribution.png", dpi=150)
    plt.show()
    print(f"Age Exact: {np.mean(np.array(age_preds)==np.array(age_true))*100:.1f}%")
    print(f"Age +-1: {np.mean(np.abs(errors)<=1)*100:.1f}%")

plot_age_distribution(age_true, age_preds)

## 16. Per-Class Accuracy

In [ ]:
for i, group in enumerate(cfg.AGE_GROUPS):
    mask = np.array(age_true) == i
    if np.sum(mask) > 0:
        acc = np.mean(np.array(age_preds)[mask] == i) * 100
        print(f"{group}: {acc:.1f}%")

## 17. Sample Predictions

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
indices = np.random.choice(len(val_dataset), 16, replace=False)
for idx, ax in zip(indices, axes.flatten()):
    img, g_true, a_true, _ = val_dataset[idx]
    with torch.no_grad():
        g_logits, a_logits = model(img.unsqueeze(0).to(device))
        g_pred, a_pred = g_logits.argmax(1).item(), a_logits.argmax(1).item()
    img_np = img.numpy().transpose(1,2,0)
    img_np = img_np * [0.229,0.224,0.225] + [0.485,0.456,0.406]
    ax.imshow(np.clip(img_np, 0, 1))
    color = "green" if g_pred==g_true and a_pred==a_true else "red"
    ax.set_title(f"{["M","F"][g_pred]} {cfg.AGE_GROUPS[a_pred]}", color=color)
    ax.axis("off")
plt.savefig(f"{cfg.OUTPUT_DIR}/sample_predictions.png", dpi=150)
plt.show()

## 18. Model Summary

In [ ]:
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")
print(f"Size: {total*4/1e6:.2f} MB")

## 19. Final Summary

In [ ]:
summary = {
    "Gender Acc": f"{np.mean(np.array(gender_preds)==np.array(gender_true))*100:.1f}%",
    "Age Exact": f"{np.mean(np.array(age_preds)==np.array(age_true))*100:.1f}%",
    "Age +-1": f"{np.mean(np.abs(np.array(age_preds)-np.array(age_true))<=1)*100:.1f}%",
    "Params": f"{sum(p.numel() for p in model.parameters()):,}"
}
print(pd.DataFrame([summary]).T)
pd.DataFrame([summary]).T.to_csv(f"{cfg.OUTPUT_DIR}/final_summary.csv")
print("Saved final_summary.csv")

## 📦 All Output Files

In [ ]:
for f in sorted(os.listdir(cfg.OUTPUT_DIR)):
    if os.path.isfile(os.path.join(cfg.OUTPUT_DIR, f)):
        print(f"  {f}: {os.path.getsize(os.path.join(cfg.OUTPUT_DIR, f))/1e6:.2f} MB")